In [1]:
from __future__ import annotations

import os
from typing import Any, Dict, List, Tuple
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from agent.prompts import PROMPTS
from agent.query_state import (
    QueryState,
    ResponseGeneration,
    Reference,
    FALLBACK_CONFIDENCE_THRESHOLD,
)
from langchain.chat_models import init_chat_model
from agent.config import get_attr_safe

/Users/jajajou1778/UIT_DOCS_AGENT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model=os.getenv("LLM_MODEL","Qwen/Qwen3-4B-Instruct-2507")

In [3]:
model

'Qwen/Qwen3-4B-Instruct-2507'

In [4]:
base_url=os.getenv("OPENAI_BASE_URL"),
base_url

('https://router.huggingface.co/v1',)

In [5]:
llm = init_chat_model(
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    model=os.getenv("LLM_MODEL","Qwen/Qwen3-4B-Instruct-2507"),
    streaming=False,
    temperature=float(os.getenv("AGENT3_TEMPERATURE", "0.3")),
    model_kwargs={"tool_choice": "none"}
)

In [6]:
prompt_text = PROMPTS["response_generation_prompt"]

In [7]:
llm_json = llm.bind(response_format={"type": "json_object"})
llm_structured_output = llm_json.with_structured_output( #type: ignore
    ResponseGeneration,          
    method="json_schema",       
    include_raw=False             
) 

In [8]:

msgs = [
    SystemMessage(content=prompt_text),
    HumanMessage(content="Generate response cho query trên.")
]

response_gen = llm_structured_output.invoke(input=msgs)


In [9]:
response_gen

ResponseGeneration(response_text='Theo [Quy chế đào tạo 2024](https://daa.uit.edu.vn/quy-che-2024.pdf), sinh viên ngành Khoa học Máy tính cần tích lũy tối thiểu **140 tín chỉ** để đủ điều kiện tốt nghiệp.\n\nCụ thể, 140 tín chỉ này bao gồm:\n- Kiến thức giáo dục đại cương: 40 tín chỉ\n- Kiến thức cơ sở ngành: 50 tín chỉ\n- Kiến thức chuyên ngành: 45 tín chỉ\n- Thực tập và khóa luận: 5 tín chỉ\n\nNgoài ra, sinh viên cũng cần đạt các điều kiện khác như GPA tối thiểu 2.0, hoàn thành chương trình giáo dục thể chất và giáo dục quốc phòng.', response_type='full_answer', references=[Reference(title='Quy chế đào tạo 2024', url='https://daa.uit.edu.vn/quy-che-2024.pdf', relevance=0.95, excerpt='Điều 15: Điều kiện tốt nghiệp...')])

In [12]:
get_attr_safe(response_gen,"response_text")

'Theo [Quy chế đào tạo 2024](https://daa.uit.edu.vn/quy-che-2024.pdf), sinh viên ngành Khoa học Máy tính cần tích lũy tối thiểu **140 tín chỉ** để đủ điều kiện tốt nghiệp.\n\nCụ thể, 140 tín chỉ này bao gồm:\n- Kiến thức giáo dục đại cương: 40 tín chỉ\n- Kiến thức cơ sở ngành: 50 tín chỉ\n- Kiến thức chuyên ngành: 45 tín chỉ\n- Thực tập và khóa luận: 5 tín chỉ\n\nNgoài ra, sinh viên cũng cần đạt các điều kiện khác như GPA tối thiểu 2.0, hoàn thành chương trình giáo dục thể chất và giáo dục quốc phòng.'